In [1]:
import pickle
import os
import torch
import contextlib
import json
from mylib import yolo_patch
from mylib import utils
import math
import numpy as np
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
import skimage.io as io
from ultralytics import YOLO 
from dotenv import load_dotenv
from tqdm import tqdm  # For progress tracking
import time
load_dotenv()  # This loads from .env in the current directory

True

In [2]:
# === Load YOLO model ===
yolo_model = YOLO("yolo11x.pt")  # Swap with yolov8s.pt or yolov8x.pt as needed

In [3]:
# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
yolo_model = yolo_model.to(device)

Using device: cuda:0


In [4]:
# Load annotations for both datasets
cocos = {
    "train2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_train2017.json")),
    "val2017": COCO(os.path.join(os.getenv("COCO_DATA"), "annotations/instances_val2017.json")),
}

loading annotations into memory...
Done (t=13.50s)
creating index...
index created!
loading annotations into memory...
Done (t=0.46s)
creating index...
index created!


In [5]:
# YOLO ID -- Class mapping from json file
with open('indoor_objects.json') as f:
    indoor_objects = json.load(f)
    indoor_objects = {k: v for d in indoor_objects['indoor_classes'] for k, v in d.items()}

# Bin range -- Class from json file
with open("out_bins_per_class.json") as f:
    out_bins_per_class = json.load(f)

# YOLO IDs -- COCO IDs mapping
train_coco = list(cocos.values())[0]
all_cats = train_coco.loadCats(train_coco.getCatIds())
yolo_id_to_coco_id = {i: cat["id"] for i, cat in enumerate(sorted(all_cats, key=lambda x: x["id"]))} # eg. Yolo ID 79 -> COCO ID 90

# Build lists 
yolo_ids = [int(k) for k in indoor_objects.keys()]
coco_ids = [yolo_id_to_coco_id[int(k)] for k in indoor_objects.keys()]
classes_names = [v for v in indoor_objects.values()]
classes_bins = [out_bins_per_class[v] for v in indoor_objects.values()]
num_classes = len(classes_bins)

# Print information
print("Number of classes:", num_classes,'\n')
print("YOLO IDs:", yolo_ids)
print("COCO IDs:", coco_ids)
print("Classes names:", classes_names)
print("Classes bins:", classes_bins)


Number of classes: 27 

YOLO IDs: [26, 39, 40, 41, 42, 43, 44, 45, 47, 56, 57, 58, 59, 60, 61, 62, 63, 65, 67, 68, 69, 70, 71, 73, 74, 75, 77]
COCO IDs: [31, 44, 46, 47, 48, 49, 50, 51, 53, 62, 63, 64, 65, 67, 70, 72, 73, 75, 77, 78, 79, 80, 81, 84, 85, 86, 88]
Classes names: ['handbag', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'apple', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'remote', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'book', 'clock', 'vase', 'teddy bear']
Classes bins: [[0.0, 0.05, 0.11, 0.19, 0.33, 0.53, 0.84, 1.41, 2.44, 4.87, 99.4], [0.0, 0.06, 0.11, 0.17, 0.25, 0.37, 0.55, 0.87, 1.52, 3.39, 99.01], [0.0, 0.09, 0.17, 0.26, 0.41, 0.65, 1.03, 1.73, 3.2, 7.09, 98.2], [0.0, 0.07, 0.13, 0.22, 0.34, 0.54, 0.89, 1.55, 2.78, 5.78, 100.0], [0.0, 0.11, 0.23, 0.4, 0.62, 0.99, 1.64, 2.71, 4.88, 9.2, 99.45], [0.0, 0.04, 0.09, 0.17, 0.3, 0.55, 0.94, 1.59, 3.0, 6.7, 83.22], [0.0, 0.06, 0.12, 0.21, 0.34, 0.54, 

In [6]:
# === Helper: Get COCO split for a given image ID ===
def find_split_for_image(img_id):
    for split, coco_obj in cocos.items():
        if img_id in coco_obj.imgs:
            return split, coco_obj
    return None, None

In [ ]:
# === IoU threshold for filtering ===
IOU_THRESHOLD = 0.5

# === Data structure to store the vectors scores ===
vector_scores_struct = {cls: {b: [] for b in range(len(classes_bins[0])-1)} for cls in classes_names} 

# Loop through every class_id
for i in range(num_classes):

    # Get all image IDs for this class (from both splits)
    img_ids = []
    for split, coco_obj in cocos.items():
        img_ids += coco_obj.getImgIds(catIds=[coco_ids[i]])

    k = 0
    # Loop through each image ID
    for img_id in tqdm(img_ids, desc=f"Processing {len(img_ids)} images for {classes_names[i]}", unit="image"):
        
        # Find the split for the current image ID
        split, any_coco = find_split_for_image(img_id)
        
        # Load the image 
        img_data = any_coco.imgs[img_id]
        img_path = os.path.join(os.getenv("COCO_DATA"), split, img_data["file_name"])
        img = io.imread(img_path)

        # Plot ground truth boxes from COCO using COCO API
        ann_ids = any_coco.getAnnIds(imgIds=img_id, catIds=[coco_ids[i]], iscrowd=None)
        anns = any_coco.loadAnns(ann_ids)

        if img.ndim == 2:  # grayscale
            img = np.stack([img]*3, axis=-1)

        # Prediction
        results = yolo_model.predict(source=img, device='cuda:0', conf=0.25, iou=0.45, verbose=False, max_det=100)
        
        # Unpack results
        result = results[0] # One image, one result
        xyxy = result.boxes.xyxy
        xywh = result.boxes.xywh
        conf = result.boxes.conf
        cls = result.boxes.cls
        prob_vectors = result.boxes.data[:, 6:]  # Probability vectors for each class

        # Plot the image
        # plt.figure(figsize=(5, 5))
        # plt.imshow(img)
        # plt.axis('off')
        # plt.title(f"Image ID: {img_id}, Ground-truth Class: {classes_names[i]}")

        # Loop ground truth annotations
        for ann in anns:
            bbox = ann["bbox"]
            x1_gt, y1_gt, w_gt, h_gt = bbox
            x2_gt = x1_gt + w_gt
            y2_gt = y1_gt + h_gt

            # Compute the scale of ground truth bounding box compared to whole image
            scale_gt = w_gt * h_gt / (img.shape[0] * img.shape[1])*100
            # print(f"Ground truth Scale: {scale_gt:.4f}%")

            # Compute bin index
            bin_idx = utils.bin_index(scale_gt, classes_bins[i])

            # print(f"Ground truth Bin index: {bin_idx}")
            # print("Classes bins:", classes_bins[i])

            # Plot ground truth boxes
            # plt.gca().add_patch(plt.Rectangle((x1_gt, y1_gt), w_gt, h_gt, fill=False, edgecolor="green", linewidth=2))

            # Loop through every detection
            for box_xyxy, box_xywh, c, cl, prob_v in zip(xyxy, xywh, conf, cls, prob_vectors):
                x1, y1, x2, y2 = box_xyxy.tolist()
                cx, cy, w, h = box_xywh.tolist()
                confidence = c.item()
                class_id = int(cl.item()) 
                prob_vector = prob_v.tolist()

                # Normalize probability vector
                prob_vector = [x / sum(prob_vector) for x in prob_vector]

                # Compute IoU
                IoU = utils.compute_iou([x1, y1, x2, y2], [x1_gt, y1_gt, x2_gt, y2_gt])

                # Check if IoU is above the threshold
                if IoU > IOU_THRESHOLD:

                    # Add vector probability to the corresponding bin
                    vector_scores_struct[classes_names[i]][bin_idx].append(prob_vector)

                    # Print results
                    # print(f"XYXY (corner format): ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})")
                    # print(f"XYWH (center format): ({cx:.1f}, {cy:.1f}, {w:.1f}, {h:.1f})")
                    # print(f"Confidence: {confidence:.4f}, Class ID: {class_id}, Class Name: {yolo_model.names[class_id]}")
                    # print(f"Probability vector: {[f'{x:.3e}' for x in prob_vector]}")
                    # print(f"IoU: {IoU:.4f}")
                    # print("---")

                    # # Plot predicted boxes
                    # plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="red", linewidth=1))
                    # plt.text(x1, y1, f"{yolo_model.names[class_id]} {confidence:.2f}", color="red", fontsize=12)
                
        # plt.show()

        # # Increment
        # k +=1

        # if k > 5:
        #     break


Processing 7133 images for handbag:   0%|          | 2/7133 [00:00<09:54, 11.99image/s]

[2.528543115372401e-05, 9.106783890069797e-07, 7.128828009617144e-07, 8.24246739280969e-07, 1.4194701540768503e-06, 2.210471652240924e-05, 1.2432394664540737e-05, 7.936998400975336e-05, 1.235529191004441e-06, 5.5391310241696894e-06, 2.2377243641537e-05, 1.9697415099386546e-05, 5.703091760963488e-06, 1.5165583663619665e-05, 3.739568930602159e-06, 2.146142777933999e-06, 8.163581782254535e-06, 5.2683559164955044e-05, 3.141580662677591e-05, 4.325907264505726e-05, 0.9991113605244484, 2.209232565504639e-05, 4.3957210813336215e-05, 1.5858668335099323e-05, 1.2664572400286784e-06, 2.2443688081746832e-05, 9.90485313481137e-07, 1.8180500374860738e-06, 9.147670982744166e-06, 3.805626807186676e-06, 2.9554706586729055e-07, 1.451195475783307e-07, 7.14440027431155e-06, 7.611577920660359e-06, 1.3985203466880547e-05, 8.496367376707404e-06, 4.529486488263893e-06, 1.0711211719264821e-06, 2.9677559489525394e-06, 2.579897643227769e-07, 1.1259331362092312e-07, 1.1822534087904646e-05, 3.148200913923003e-07, 4

Processing 7133 images for handbag:   0%|          | 4/7133 [00:00<09:48, 12.12image/s]

[0.9989949757707755, 5.346705133081293e-06, 2.4095674662059684e-05, 1.2696949452706323e-05, 1.5150742757991963e-05, 9.795846349271162e-06, 1.5162306627816084e-05, 1.3869337547685072e-05, 1.007081248049954e-05, 8.08671652877616e-06, 9.193596146160438e-06, 6.226972011451662e-06, 7.663234096470084e-06, 5.480502728199971e-06, 1.4556142573816244e-05, 3.191099851265834e-05, 3.4846012000456237e-05, 2.8013334144293382e-05, 1.1816316586305613e-05, 1.704719713012765e-05, 1.9014242771408473e-05, 1.468839172089626e-05, 9.498356020660934e-06, 1.9310314534275768e-05, 1.658948805731124e-05, 1.5278455198563943e-05, 1.1022789836793853e-05, 1.1778502973397643e-05, 5.030321034530036e-06, 3.2852225091269814e-06, 1.0176937421287119e-05, 8.565195663731367e-06, 4.501569242953223e-06, 1.3462422592008435e-05, 4.641337561004889e-06, 4.3026730961474084e-06, 5.6434972839977555e-06, 1.120913858129823e-05, 4.277278451746588e-06, 9.619350311557224e-06, 6.916118781576444e-06, 6.420311706736141e-06, 2.8465614716548934

Processing 7133 images for handbag:   0%|          | 8/7133 [00:00<10:25, 11.40image/s]

...

Processing 7133 images for handbag:   1%|          | 38/7133 [00:03<09:31, 12.41image/s]

[0.999312465100843, 7.404491752046467e-06, 3.442864904257899e-05, 1.2179491691525277e-05, 7.537358563998009e-06, 1.3571659650152607e-05, 1.1494975956681297e-05, 9.2563318994741e-06, 6.8886825470815745e-06, 5.64993712132597e-06, 5.13469812625848e-06, 3.1861453896592767e-06, 7.5362804028228485e-06, 6.779297117057608e-06, 5.154981095130687e-06, 1.5924159899213677e-05, 1.1835930840134957e-05, 1.2635226763806936e-05, 3.898182399083495e-06, 5.775179722950873e-06, 7.58826387964812e-06, 4.368504517713685e-06, 8.351702297215512e-06, 4.469226117286111e-06, 1.3055092965315341e-05, 1.0903363917599757e-05, 1.2717807387492635e-05, 5.7226028127748375e-06, 4.796333399028557e-06, 2.8248321846389956e-06, 5.782873603637592e-06, 4.316950921833597e-06, 3.1997535921544663e-06, 4.334384323566872e-06, 3.824031888967645e-06, 3.645442791451622e-06, 4.038573856981842e-06, 7.643170052784608e-06, 4.93963544739541e-06, 1.1101129905257191e-05, 6.472690709356135e-06, 8.777415867810465e-06, 3.5913568509231846e-06, 4.4

Processing 7133 images for handbag:   1%|          | 40/7133 [00:03<09:59, 11.84image/s]

[0.9998116741537716, 8.118925386898813e-07, 6.285685011123903e-06, 1.9536132496855846e-06, 3.086120791346952e-06, 3.944970046887299e-06, 1.2325664401881462e-06, 4.121018458452684e-06, 2.312832564998258e-06, 2.8090375712108305e-06, 3.491203781914616e-06, 1.3912061361137776e-06, 2.2981739268669322e-06, 8.825719018410398e-07, 4.040163299522608e-06, 4.499996320891186e-06, 5.1590758841691275e-06, 2.603034270612692e-06, 9.892388324018833e-07, 2.6973679378079354e-06, 4.525232180208772e-06, 2.5835336984355147e-06, 1.8101395757291695e-06, 4.1675721360005525e-06, 1.925296340187239e-06, 3.080953843626621e-06, 1.5139756444728178e-06, 1.156274756300158e-06, 6.944995361422799e-07, 7.490162349335732e-07, 8.92755705023319e-07, 1.1051016139595569e-06, 1.3408389482325804e-06, 2.2038834624061113e-06, 1.7407378589610908e-06, 1.1192878773653696e-06, 1.498262139576003e-06, 1.180598144071747e-06, 1.0629382280787177e-06, 7.741739817209912e-06, 4.581521867079545e-06, 3.985283412226851e-06, 9.120747139847113e-0

KeyboardInterrupt: 

In [ ]:
# Save to pickle
with open("vector_scores_struct.pkl", "wb") as f:
    pickle.dump(vector_scores_struct, f)

In [ ]:
# Convert the vector scores structure to numpy arrays
for cls in vector_scores_struct:
    for b in vector_scores_struct[cls]:
        vector_scores_struct[cls][b] = np.vstack(vector_scores_struct[cls][b]) if vector_scores_struct[cls][b] else np.empty((0, 80))

In [ ]:
# Print the number of vectors in each bin
for cls in vector_scores_struct:
    for b in vector_scores_struct[cls]:
        count = len(vector_scores_struct[cls][b]) if isinstance(vector_scores_struct[cls][b], list) else vector_scores_struct[cls][b].shape[0]
        print(f"{cls} - bin {b}: {count} vectors")


In [ ]:
# Print the whole structure
print(vector_scores_struct)

In [ ]:
# eng = matlab.engine.start_matlab()
# eng.addpath(os.getenv("FASTFIT_TOOLBOX"), nargout=0)

# eng.quit()